## **Exercise 1: Image Classification with a Pre-Trained Model**

Load a pre-trained model (e.g., ResNet18 from torchvision), pass an image through it, and print the top-5 predicted classes with confidence scores.

**Skills practiced:** Transfer learning, model loading, basic PyTorch workflow

## **Neural networks need an image in a tensor of a specific size, scaled and normalized the same as the data it was trained on**

What I should know:
- The shape of the pipeline
  - resize -> crop -> tensor -> normalize
- Why normalization matters
- Why the crop size is 224
- That the mean and std should match the training data

What is acceptable to look up:
- The literal mean/std triplet values
- Exact resize/crop dimensions per model architecture
- Exact API syntax

## **On batch dimension**

PyTorch models expect a batch of inputs as such: (batch_size, channels, height, width)

After preprocessing, a single image has 3-dimensions, but the model expects 4: 

`input_tensor.shape  # torch.Size([3, 224, 224])`

Whenever feeding a tensor into a model, documentation (or error messages) will supply expected dimensions. `squeeze`/`unsqueeze`/`view`/`reshape` are the tools to fix this. Printing shape constantly helps with this:

```
print(input_tensor.shape)   # torch.Size([3, 224, 224])
print(input_batch.shape)    # torch.Size([1, 3, 224, 224])
print(output.shape)         # torch.Size([1, 1000])
```

In [ ]:
import torch                                # Core deep learning library, includes tensors, autogradient, etc.
from torchvision import models, transforms  # CV library, contains pretrained models and image transform functions
from PIL import Image                       # Used for loading images into memory
import json                                 # Used to decode classification labels
import urllib.request                       # Used to decode classification labels

### ---------------------------
# Get model in inference mode
### ---------------------------
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT) # Load the specific model into memory
model.eval()                                                     # Switches the model out of training mode and into inference mode

### ------------------------------
# Preprocess image for inference
### ------------------------------

# Preprocess the image for the model manually
# preprocess = transforms.Compose([
#     transforms.Resize(256),
#     transforms.CenterCrop(224),
#     transforms.ToTensor(),
#     transforms.Normalize(
#         mean=[0.485, 0.456, 0.406],
#         std=[0.229, 0.224, 0.225]
#     )
# ])

# Auto matches the training preprocessing
weights = models.ResNet18_Weights.DEFAULT
preprocess = weights.transforms()

# Apply preprocessing to the image
img = Image.open("1_data/kittens/kitten1.jpg").convert("RGB")
input_tensor = preprocess(img)

# Add a batch dimension
input_batch = input_tensor.unsqueeze(0) 

### --------------------------------------------------------
# Run the image through the model and get top 5 predictions
### --------------------------------------------------------

# no_grad disables gradient tracking to save computation on inference
with torch.no_grad(): 
    # Outputs raw logits, need to transform to probabilities with softmax
    output = model(input_batch)

# Transform logits into probability distribution
probabilities = torch.nn.functional.softmax(output[0], dim=0)
# print(probabilities)

# Get top 5 probabilities
top_5_prob, top_5_idx = torch.topk(probabilities, 5)

### ----------------------------------------
# Get class names from model documentation
### ----------------------------------------

url = 'https://gist.githubusercontent.com/ageitgey/4e1342c10a71981d0b491e1b8227328b/raw/24d78ea09a31fdff540a8494886e0051e3ad68f8/imagenet_classes.txt'
categories = urllib.request.urlopen(url).read().decode("utf-8").split("\n")
categories = categories[4:]
categories = [line.split(", ", 1)[1] for line in categories if line.strip()]
# print(categories)

### ----------------
# Print the output
### ----------------

for prob, idx in zip(top_5_prob, top_5_idx):
    print(f"{categories[idx]}: {prob.item()*100:.2f}%")